In [18]:
import os
import numpy as np
import pandas as pd

TSV_PATH = "../data/gaps_openai.tsv"   # <- change
OUT_DIR = "../artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

gaps = pd.read_csv(TSV_PATH, sep="\t")

# Clean text columns
for c in ["gap_sentence", "paragraph_text"]:
    gaps[c] = gaps[c].astype(str).str.replace("\n", " ").str.strip()

gaps["confidence"] = pd.to_numeric(gaps["confidence"], errors="coerce").fillna(0.0)

# Filter
gaps = gaps[(gaps["confidence"] >= 0.5) & (gaps["gap_sentence"].str.len() >= 20)]
gaps = gaps.drop_duplicates(subset=["id", "gap_sentence"]).reset_index(drop=True)

print("Loaded gaps:", gaps.shape)
display(gaps.head(3))

# Save cleaned
gaps.to_csv(f"{OUT_DIR}/gaps_clean.tsv", sep="\t", index=False)


Loaded gaps: (62, 5)


,id,gap_type,gap_sentence,paragraph_text,confidence
0,2512.19725,future_work,"In particular, our study highlights the need f...",These findings provide concrete guidance for p...,0.95
1,2503.17793,future_work,The future work will focus on further pushing ...,The future work will focus on further pushing ...,0.95
2,2503.17793,future_work,We plan to improve reasoning performance with ...,The future work will focus on further pushing ...,0.95


In [19]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = gaps["gap_sentence"].tolist()
X = embedder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print("Embeddings:", X.shape)
np.save(f"{OUT_DIR}/gap_embeddings.npy", X)


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.70it/s]

Embeddings: (62, 384)


In [21]:
n = len(gaps)
print("n gaps:", n)

if n < 8:
    raise ValueError("Too few gaps to cluster. Extract more papers/items first.")

if n < 150:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    k_candidates = list(range(4, min(12, n)))  # 2..11 or up to n-1
    best_k, best_s = None, -1

    for k in k_candidates:
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        lbl = km.fit_predict(X)
        s = silhouette_score(X, lbl)
        if s > best_s:
            best_k, best_s = k, s

    km = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
    gaps["cluster_id"] = km.fit_predict(X)

    print("KMeans chosen k:", best_k, "| silhouette:", round(best_s, 3))

else:
    !pip -q install hdbscan
    import hdbscan

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=max(8, n // 40),
        min_samples=2,
        metric="euclidean"
    )
    gaps["cluster_id"] = clusterer.fit_predict(X)

    print("HDBSCAN clusters:", gaps["cluster_id"].nunique())

print(gaps["cluster_id"].value_counts().head(15))

# Save
gaps.to_csv(f"{OUT_DIR}/gaps_with_clusters.tsv", sep="\t", index=False)


n gaps: 62
KMeans chosen k: 10 | silhouette: 0.026
cluster_id
5    11
2    10
9     9
1     8
4     7
7     6
8     3
3     3
6     3
0     2
Name: count, dtype: int64


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

def keyword_label(sentences, k=6):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=2000)
    M = v.fit_transform(sentences)
    scores = np.asarray(M.mean(axis=0)).ravel()
    terms = np.array(v.get_feature_names_out())
    top = terms[np.argsort(scores)[::-1][:k]]
    return ", ".join(top)

cluster_labels = {}
for cid in sorted(gaps["cluster_id"].unique()):
    if cid == -1:
        continue
    sents = gaps[gaps.cluster_id == cid]["gap_sentence"].tolist()
    cluster_labels[cid] = keyword_label(sents, k=6)

cluster_labels_df = pd.DataFrame(
    [{"cluster_id": cid, "theme_label": lbl} for cid, lbl in cluster_labels.items()]
).sort_values("cluster_id")

display(cluster_labels_df.head(10))

cluster_labels_df.to_csv(f"{OUT_DIR}/cluster_labels.tsv", sep="\t", index=False)


,cluster_id,theme_label
0,0,"high, main, main paper, paper, implementation,..."
1,1,"research, compression, future, investigate, di..."
2,2,"deep, learning, deep learning, dnn, networks, ..."
3,3,"market, order derivatives, order, derivatives,..."
4,4,"work, future, future work, patient, datasets, ..."
5,5,"learning, model, future, domain, performance, ..."
6,6,"machine, training, end, svm, approach, support"
7,7,"future, improving, work, future work, efficien..."
8,8,"future, worth, worth pursuing, inquiry, lines ..."
9,9,"limited, bias, data, analysis, paper, paper li..."


In [23]:
def top_examples(df, cid, n=3):
    ex = df[df.cluster_id == cid].sort_values("confidence", ascending=False).head(n)
    return ex[["id","gap_type","confidence","gap_sentence"]].to_dict("records")

cluster_summary = (
    gaps[gaps.cluster_id != -1]
    .groupby("cluster_id")
    .agg(
        n_items=("gap_sentence","count"),
        n_papers=("id","nunique"),
        avg_conf=("confidence","mean"),
    )
    .reset_index()
)

cluster_summary["theme_label"] = cluster_summary["cluster_id"].map(cluster_labels)
cluster_summary["examples"] = cluster_summary["cluster_id"].apply(lambda cid: top_examples(gaps, cid, 3))

cluster_summary = cluster_summary.sort_values(
    ["n_papers","n_items","avg_conf"],
    ascending=False
).reset_index(drop=True)

display(cluster_summary.head(15))

cluster_summary.to_csv(f"{OUT_DIR}/cluster_summary.tsv", sep="\t", index=False)


,cluster_id,n_items,n_papers,avg_conf,theme_label,examples
0,5,11,10,0.950909,"learning, model, future, domain, performance, ...","[{'id': 2504.16732, 'gap_type': 'open_problem'..."
1,2,10,9,0.947000,"deep, learning, deep learning, dnn, networks, ...","[{'id': 2102.03018, 'gap_type': 'future_work',..."
2,9,9,9,0.944444,"limited, bias, data, analysis, paper, paper li...","[{'id': 2508.13182, 'gap_type': 'open_problem'..."
3,1,8,6,0.941250,"research, compression, future, investigate, di...","[{'id': 2401.08233, 'gap_type': 'future_work',..."
4,4,7,6,0.935714,"work, future, future work, patient, datasets, ...","[{'id': 2308.13569, 'gap_type': 'future_work',..."
5,7,6,6,0.941667,"future, improving, work, future work, efficien...","[{'id': 2503.17793, 'gap_type': 'future_work',..."
6,8,3,3,0.973333,"future, worth, worth pursuing, inquiry, lines ...","[{'id': 2106.10464, 'gap_type': 'future_work',..."
7,3,3,3,0.963333,"market, order derivatives, order, derivatives,...","[{'id': 2310.13369, 'gap_type': 'limitation', ..."
8,6,3,2,0.933333,"machine, training, end, svm, approach, support","[{'id': 2506.00483, 'gap_type': 'limitation', ..."
9,0,2,1,0.950000,"high, main, main paper, paper, implementation,...","[{'id': 2112.11427, 'gap_type': 'limitation', ..."


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

# Build centroids
centroids = {}
for cid in sorted(gaps["cluster_id"].unique()):
    if cid == -1:
        continue
    idx = np.where(gaps["cluster_id"].values == cid)[0]
    centroids[cid] = X[idx].mean(axis=0)

cids = sorted(centroids.keys())
print("clusters for linking:", len(cids))

pairs_df = pd.DataFrame(columns=["cluster_a","cluster_b","cosine_sim","label_a","label_b"])

if len(cids) >= 2:
    C = np.stack([centroids[c] for c in cids])
    S = cosine_similarity(C, C)

    pairs = []
    for i, ca in enumerate(cids):
        for j in range(i+1, len(cids)):
            cb = cids[j]
            pairs.append((ca, cb, float(S[i, j])))

    pairs = sorted(pairs, key=lambda x: x[2], reverse=True)[:30]
    pairs_df = pd.DataFrame(pairs, columns=["cluster_a","cluster_b","cosine_sim"])
    pairs_df["label_a"] = pairs_df["cluster_a"].map(cluster_labels)
    pairs_df["label_b"] = pairs_df["cluster_b"].map(cluster_labels)

display(pairs_df.head(15))
pairs_df.to_csv(f"{OUT_DIR}/cluster_pairs.tsv", sep="\t", index=False)


clusters for linking: 10


,cluster_a,cluster_b,cosine_sim,label_a,label_b
0,2,5,0.609657,"deep, learning, deep learning, dnn, networks, ...","learning, model, future, domain, performance, ..."
1,1,9,0.573678,"research, compression, future, investigate, di...","limited, bias, data, analysis, paper, paper li..."
2,1,5,0.561762,"research, compression, future, investigate, di...","learning, model, future, domain, performance, ..."
3,1,2,0.558066,"research, compression, future, investigate, di...","deep, learning, deep learning, dnn, networks, ..."
4,1,4,0.539070,"research, compression, future, investigate, di...","work, future, future work, patient, datasets, ..."
5,4,9,0.539017,"work, future, future work, patient, datasets, ...","limited, bias, data, analysis, paper, paper li..."
6,5,9,0.503383,"learning, model, future, domain, performance, ...","limited, bias, data, analysis, paper, paper li..."
7,4,8,0.483266,"work, future, future work, patient, datasets, ...","future, worth, worth pursuing, inquiry, lines ..."
8,4,5,0.475118,"work, future, future work, patient, datasets, ...","learning, model, future, domain, performance, ..."
9,3,5,0.463987,"market, order derivatives, order, derivatives,...","learning, model, future, domain, performance, ..."


In [25]:
def evidence_for_cluster(df, cid, n=5):
    cols = ["id","gap_type","confidence","gap_sentence","paragraph_text"]
    return df[df.cluster_id == cid].sort_values("confidence", ascending=False).head(n)[cols]

if len(pairs_df) > 0:
    ca = int(pairs_df.iloc[0]["cluster_a"])
    cb = int(pairs_df.iloc[0]["cluster_b"])

    print("Top pair:", ca, "<->", cb, "| sim:", pairs_df.iloc[0]["cosine_sim"])
    print("A:", pairs_df.iloc[0]["label_a"])
    print("B:", pairs_df.iloc[0]["label_b"])

    display(evidence_for_cluster(gaps, ca, n=5))
    display(evidence_for_cluster(gaps, cb, n=5))


Top pair: 2 <-> 5 | sim: 0.6096569299697876
A: deep, learning, deep learning, dnn, networks, future
B: learning, model, future, domain, performance, multi


,id,gap_type,confidence,gap_sentence,paragraph_text
3,2102.03018,future_work,0.99,We plan to extend this task by performing simi...,6. Future Work and Challenges We plan to exten...
32,2402.07506,future_work,0.98,"In the future, the NeuralSentinel tool will be...",This work presents the NeuralSentinel tool tha...
5,2401.17544,future_work,0.95,Users can also define customized gradient func...,One may notice that the casting function lever...
6,2207.09511,future_work,0.95,One avenue involves exploring the connections ...,"Finally, we note some potential research direc..."
21,2504.03738,future_work,0.95,An important future research direction is to e...,"Among the methods discussed in Section 3.2, cr..."


,id,gap_type,confidence,gap_sentence,paragraph_text
39,2504.16732,open_problem,0.98,"Notwithstanding these contributions, the devel...","limitations, advanced merging strategies such ..."
4,2102.03018,limitation,0.95,Most of the challenges got raised due to many ...,We faced many challenges when ﬁnishing this ta...
2,2503.17793,future_work,0.95,We plan to improve reasoning performance with ...,The future work will focus on further pushing ...
16,2310.13369,future_work,0.95,"As our future revenue, we envision new adaptiv...",Limitations and Future Directions. Our current...
17,2205.12753,future_work,0.95,This phenomenon indicates that the performance...,Tab. S12 reports the result of 6 domain genera...
